# Person 1 cloud pilot — bounded GPU run

Run **Runtime → Run all** on a free **T4 GPU** runtime.

This executes the Person 1 trainer (`src/train.py`) through `scripts/run_pilot.py` for the three
variants (MLM C, latent A, latent B), then frozen geometry via `src/evaluate_bridge.py`.

It is a **bounded pilot**, not the full proposed 5M-token study: a few hundred WikiText-2 records
and a segment cap of 50 updates per variant. No result here supports a collapse or transfer conclusion
beyond the measured pilot.

Branch: `integration/person1-person2` (Person 1 trainer + Person 2 data/evaluation modules).

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))
    print('vram_gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
assert torch.cuda.is_available(), 'No CUDA device: switch the runtime to T4 GPU before continuing'

In [ ]:
%%bash
set -e
cd /content
rm -rf 7th-sem
git clone -q --branch integration/person1-person2 --depth 1 https://github.com/dakshadev/7th-sem.git 7th-sem
cd 7th-sem
echo "COMMIT $(git rev-parse HEAD)"
pip install -q -r requirements.txt 2>&1 | tail -3
pip install -q -e . --no-deps 2>&1 | tail -3
python -c "import src.data, src.evaluate; print('person2 modules import OK')"
python -c "import torch; print('post-install torch', torch.__version__, 'cuda', torch.cuda.is_available())"
OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 python -m pytest -q 2>&1 | tail -8

In [ ]:
%%bash
set -e
cd /content/7th-sem
mkdir -p /content/artifacts
python scripts/build_batches.py \
  --dataset Salesforce/wikitext \
  --config-name wikitext-2-raw-v1 \
  --split train \
  --tokenizer bert-base-uncased \
  --max-length 128 \
  --policy independent \
  --mask-ratio 0.15 \
  --max-records 400 \
  --batch-size 8 \
  --seed 17 \
  --out /content/artifacts/batches_independent.pt
python scripts/build_batches.py \
  --dataset Salesforce/wikitext \
  --config-name wikitext-2-raw-v1 \
  --split train \
  --tokenizer bert-base-uncased \
  --max-length 128 \
  --policy span \
  --mask-ratio 0.15 \
  --max-records 400 \
  --batch-size 8 \
  --seed 17 \
  --out /content/artifacts/batches_span.pt
ls -l /content/artifacts
python - <<'PY'
import json, glob
for path in sorted(glob.glob('/content/artifacts/*.manifest.json')):
    m = json.load(open(path))
    print(path.split('/')[-1], json.dumps(m.get('counts', m), sort_keys=True)[:400])
PY

In [ ]:
%%bash
set -e
cd /content/7th-sem
for spec in "mlm configs/mlm_c.json batches_independent.pt pilot-mlm" \
            "latent_a configs/latent_a.json batches_span.pt pilot-latent-a" \
            "latent_b configs/latent_b.json batches_independent.pt pilot-latent-b"; do
  set -- $spec
  echo "=== PILOT $1 ==="
  python scripts/run_pilot.py \
    --variant "$1" --config "$2" \
    --batches "/content/artifacts/$3" \
    --output "/content/artifacts/$4" \
    --max-updates 50 --device auto
done
echo '=== PILOT JSON ==='
for d in pilot-mlm pilot-latent-a pilot-latent-b; do
  echo "--- $d ---"
  [ -f "/content/artifacts/$d/pilot.json" ] && cat "/content/artifacts/$d/pilot.json"
done

In [ ]:
%%bash
cd /content/7th-sem || exit 1
for spec in "pilot-mlm geometry-mlm.json" "pilot-latent-a geometry-latent-a.json" "pilot-latent-b geometry-latent-b.json"; do
  set -- $spec
  echo "=== GEOMETRY $1 ==="
  python -m src.evaluate_bridge \
    --student "/content/artifacts/$1/student.pt" \
    --dataset Salesforce/wikitext \
    --config-name wikitext-2-raw-v1 \
    --split validation \
    --tokenizer bert-base-uncased \
    --max-length 128 \
    --max-records 200 \
    --batch-size 8 \
    --device auto \
    --out "/content/artifacts/$2"
  [ -f "/content/artifacts/$2" ] && cat "/content/artifacts/$2"
done

In [ ]:
%%bash
cd /content/artifacts || exit 1
echo '=== FILES ==='
find . -maxdepth 2 -type f | sort
echo '=== SHA256 ==='
sha256sum pilot-*/checkpoint.pt pilot-*/student.pt 2>/dev/null | head -12
echo '=== METRICS TAIL ==='
for d in pilot-mlm pilot-latent-a pilot-latent-b; do
  echo "--- $d ---"; tail -2 "$d/metrics.jsonl" 2>/dev/null
done